In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_regression

In [2]:
X, y= make_regression(n_samples=75, n_features=2, noise=10, random_state=42)

Feature1 = 4 + (X[:, 0] - X[:, 0].min()) / (X[:, 0].max() - X[:, 0].min()) * 6
Feature2 = 40 + (X[:, 1] - X[:, 1].min()) / (X[:, 1].max() - X[:, 1].min()) * 60
target = 1 + (y - y.min()) / (y.max() - y.min()) * 9

df = pd.DataFrame({'Feature1': Feature1, 'Feature2': Feature2, 'target': target})

df['Feature1'] = (df['Feature1'] - df['Feature1'].mean()) / df['Feature1'].std()
df['Feature2'] = (df['Feature2'] - df['Feature2'].mean()) / df['Feature2'].std()


In [3]:
def initialize_parameters(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [4]:
def linear_forward(A_prev, W, b):
    Z = np.dot(W.T, A_prev) + b
    return Z

In [5]:
def L_layer_forward(X, parameters):
    A = X
    L = len(parameters) // 2
    for l in range(1, L+1):
        A_prev = A
        W1 = parameters['W' + str(l)]
        b1 = parameters['b' + str(l)]
        A = linear_forward(A_prev, W1, b1)
    return A, A_prev


In [6]:
def update_parameters(parameters, y, y_hat, A1, X):
    W2_old = parameters['W2'].copy()

    parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y-y_hat)*A1[0][0])
    parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y-y_hat)*A1[1][0])
    parameters['b2'][0][0] = parameters['b2'][0][0] + (0.001 * 2 * (y-y_hat))

    parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001 * 2 * (y-y_hat)*W2_old[0][0]*X[0][0])
    parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001 * 2 * (y-y_hat)*W2_old[0][0]*X[1][0])
    parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001 * 2 * (y-y_hat)*W2_old[0][0])

    parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001 * 2 * (y-y_hat)*W2_old[1][0]*X[0][0])
    parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001 * 2 * (y-y_hat)*W2_old[1][0]*X[1][0])
    parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001 * 2 * (y-y_hat)*W2_old[1][0])

In [7]:
X_sample = df[['Feature1','Feature2']].values[0].reshape(2,1).astype(np.float64)
y_sample = df[['target']].values[0][0]


In [8]:
params = initialize_parameters([2,2,1])

W1_before = params['W1'].copy()
b1_before = params['b1'].copy()
W2_before = params['W2'].copy()
b2_before = params['b2'].copy()

In [9]:
y_hat, A1 = L_layer_forward(X_sample, params)
y_hat_val = y_hat[0][0]

In [10]:
update_parameters(params, y_sample, y_hat_val, A1, X_sample)

lr = 0.001
your_dW1 = (params['W1'] - W1_before) / lr
your_db1 = (params['b1'] - b1_before) / lr
your_dW2 = (params['W2'] - W2_before) / lr
your_db2 = (params['b2'] - b2_before) / lr

In [11]:
W1_tf = tf.Variable(W1_before, dtype=tf.float64)
b1_tf = tf.Variable(b1_before, dtype=tf.float64)
W2_tf = tf.Variable(W2_before, dtype=tf.float64)
b2_tf = tf.Variable(b2_before, dtype=tf.float64)

In [12]:
x_tf = tf.constant(X_sample, dtype=tf.float64)
y_tf = tf.constant([[y_sample]], dtype=tf.float64)

In [13]:
with tf.GradientTape() as tape:
    Z1 = tf.matmul(tf.transpose(W1_tf), x_tf) + b1_tf
    Z2 = tf.matmul(tf.transpose(W2_tf), Z1) + b2_tf
    loss = tf.square(y_tf - Z2)

In [14]:
grads = tape.gradient(loss, [W1_tf, b1_tf, W2_tf, b2_tf])
tf_dW1, tf_db1, tf_dW2, tf_db2 = [g.numpy() for g in grads]

In [15]:
print("W1 diff:", np.max(np.abs(your_dW1 - (-tf_dW1))))
print("b1 diff:", np.max(np.abs(your_db1 - (-tf_db1))))
print("W2 diff:", np.max(np.abs(your_dW2 - (-tf_dW2))))
print("b2 diff:", np.max(np.abs(your_db2 - (-tf_db2))))

W1 diff: 4.440892098500626e-15
b1 diff: 0.0
W2 diff: 5.051514762044462e-15
b2 diff: 0.0
